# 氟化制冷剂吸收预测模型 Kaggle 训练及可解释性分析模板
本 Notebook 专为 Kaggle 环境设计。它能自动同步您最新的 GitHub 代码，适配 Kaggle 的 PyTorch/CUDA 环境安装依赖，一键运行数据预处理、模型训练以及基于反事实干预的可解释性物理分析。

## 💡 流程说明：每次您本地修改代码并 `git push` 后：
1. 在 Kaggle 实例中**依次运行**下面的单元格。
2. **自动同步 GitHub 单元格**在检测到已有仓库时，会自动执行强行同步（`git fetch` + `git reset --hard`），保证本地代码与 GitHub 上的最新提交 100% 一致，且不会受到任何临时脏文件的干扰。

## 1. 自动同步最新的 GitHub 仓库代码
支持可选配置 GitHub Personal Access Token (如果您后续将仓库设为私有的话)。如果是公开仓库，直接运行即可。

In [ ]:
import os
import shutil

# 1. 确保在 Kaggle 工作目录下运行
os.chdir('/kaggle/working')
print(f"当前工作路径: {os.getcwd()}")

# 2. 【可选配置】如果后续您的仓库是私有仓库，请在 Kaggle 右侧的 "Add-ons -> Secrets" 中添加名为 "GITHUB_TOKEN" 的密钥
# 如果是公开仓库，此处将自动回退到免密码克隆
github_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("GITHUB_TOKEN")
    print("🔑 检测到 Kaggle Secrets 中的 GITHUB_TOKEN，将使用 Token 进行克隆/拉取。")
except Exception:
    print("ℹ️ 未检测到 Kaggle Secrets 中的 GITHUB_TOKEN，将尝试作为公开仓库拉取。")

repo_name = "GNN"
repo_path = os.path.join('/kaggle/working', repo_name)

if github_token:
    repo_url = f"https://{github_token}@github.com/badaozhiwei-cmyk/GNN.git"
else:
    repo_url = "https://github.com/badaozhiwei-cmyk/GNN.git"

if not os.path.exists(repo_path):
    # 首次运行：克隆仓库
    print(f"🚀 正在首次克隆仓库 {repo_url}...")
    !git clone {repo_url} {repo_name}
else:
    # 后续运行：强制同步 GitHub 最新代码
    print("🔄 仓库已存在，正在强制同步 GitHub 最新提交 (清除本地所有临时修改)...")
    os.chdir(repo_path)
    !git fetch origin
    !git checkout main
    !git reset --hard origin/main
    print("✅ 强制同步完成。")

# 切换到克隆出的 GNN 根目录
os.chdir(repo_path)
print(f"🏁 已成功切入项目根目录: {os.getcwd()}")

## 2. 动态安装 PyG (PyTorch Geometric) 及其配套化学库
本单元格采用高兼容性的动态检测技术，自动识别当前 Kaggle 容器的 PyTorch 与 CUDA 版本，确保适配的二进制包可以精准安装，极具健壮性。

In [ ]:
import torch

# 动态检测宿主环境的版本信息
pyt_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda

if cuda_version is not None:
    cuda_suffix = "cu" + cuda_version.replace(".", "")
else:
    cuda_suffix = "cpu"

print(f"🔍 检测到当前 Kaggle 容器配置：PyTorch={pyt_version}, CUDA/Device={cuda_suffix}")

# 构造最适配的 whl 安装链接并安装 PyG 四大件，之后安装 torch-geometric
whl_url = f"https://data.pyg.org/whl/torch-{pyt_version}+{cuda_suffix}.html"
print(f"🌐 正在从 {whl_url} 安装配套依赖...")

# 安装核心计算图引擎依赖以及 RDKit 和 Excel 读取依赖
!pip install --no-cache-dir torch-scatter torch-sparse torch-cluster torch-spline-conv -f {whl_url}
!pip install --no-cache-dir torch-geometric
!pip install --no-cache-dir rdkit pandas numpy matplotlib networkx tqdm openpyxl

print("🎉 依赖库全部安装完成！验证导入中...")
import torch_geometric
import rdkit
print(f"✅ PyG 版本: {torch_geometric.__version__}")
print(f"✅ RDKit 版本: {rdkit.__version__}")

## 3. 生成 3-Graph 体系数据集 (预处理)
每次更改物理特征（例如添加了电负性或共价半径的桶映射）后，请运行此单元格进行数据清洗与图结构构建。

In [ ]:
# 切换到 GNN 工作根目录
os.chdir('/kaggle/working/GNN')
# 运行预处理脚本
!python prepare_tri_graph_data.py

## 4. 运行模型回归训练
您可以自由选择 GAT、GIN 或 GCN 的 Runner。例如，此处默认执行 `GIN_Runner_v2.py`。训练完成后预测图像和最强模型文件会存放在对应的 outputs/checkpoints 目录下。

In [ ]:
# 进入模型运行与训练目录
os.chdir('/kaggle/working/GNN/GNN_for_property_prediction')

# ==========================================
# 【可选 Runner 列表】：
#   - GIN_Runner_v2.py (GIN 算法，V2 版特征增强，防泄露架构)
#   - GAT_Runner.py    (GAT 算法，支持注意力分配，防泄露架构)
#   - GCN_Runner_v2.py (GCN 算法，普通图卷积，防泄露架构)
# ==========================================

print("🚀 正在启动 GIN 模型回归训练与 5 折交叉验证...")
!python GIN_Runner_v2.py

## 5. 运行反事实干预与物理规律验证 (可解释性分析)
重构后支持新 3-Head 分离池化的干预分析与可解释性模块。通过人为对输入特征添加微调扰动，检验模型的预测变化量 $\Delta y$ 是否符合真实的物理规律（如电负性变强则极性变大，从而溶解度应该增加；共价半径变大则空间位阻变大，溶解度应该降低）。

In [ ]:
# 切换到可解释性分析目录
os.chdir('/kaggle/working/GNN/Explainer_for_ionic_molecule')

# 1. 运行物理特征反事实干预实验 (Counterfactual Intervention)
print("🧪 正在运行物理量干预倾向性测试...")
!python counterfactual_explain.py --model_path ../GNN_for_property_prediction/checkpoints_v2/best_seed_1.pth

# 2. 【模式 1】：1分钟快速冒烟测试 (测试3个样本的常规归因，防止报错)
print("\n🛡️ 正在运行 GNN 归因模块冒烟测试...")
!python smoke_test_v2.py --model_path ../GNN_for_property_prediction/checkpoints_v2/best_seed_1.pth

# 3. 【模式 2】：指定样本生成热力图 (推荐)
# 运行完毕后，会在 fragment_explain_result 目录生成可视化 PNG 图片
print("\n🎨 正在为指定索引的单分子对生成归因解释图...")
!python single_explain_v2.py --sample_idx 100 --model_path ../GNN_for_property_prediction/checkpoints_v2/best_seed_1.pth

# 4. 【模式 3】：全数据集进行归因统计分析 (大概耗时 1 小时)
# 运行完成后将自动保存元素重要性贡献图与 7 大节点特征重要性柱状图
# !python fragment_explain_v2.py --model_path ../GNN_for_property_prediction/checkpoints_v2/best_seed_1.pth